# rel-f1 — GPU side

This notebook is only a runner: clone the repo, install dependencies, call the script in
`src/`. No logic lives here, otherwise the project would have two codebases.

Kaggle: Settings -> Accelerator -> GPU, then Run All.


## 1. Repo

In [ ]:
!rm -rf IIS-RDL-Project
!git clone --depth 1 https://github.com/temelkovaelena/IIS-RDL-Project.git
%cd IIS-RDL-Project
!git log --oneline -1


## 2. Dependencies

Order matters: torch first (Kaggle ships it), then the PyG stack on top.


In [ ]:
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU", torch.cuda.is_available())


In [ ]:
!pip install -q relbench torch_geometric pytorch_frame sentence-transformers


## 3. Environment check

This has to pass before training anything.


In [ ]:
import relbench, torch_frame, torch_geometric
from relbench.modeling.graph import make_pkey_fkey_graph
from relbench.modeling.utils import get_stype_proposal

print("relbench", relbench.__version__)
print("torch_frame", torch_frame.__version__)
print("torch_geometric", torch_geometric.__version__)
print("make_pkey_fkey_graph:", callable(make_pkey_fkey_graph))


## 4. Build the graph on the GPU side

This confirms the edge type names match what `src/graph_builder.py` measures locally. If they
do not, the statistics in `data/graph_stats.json` describe a different graph from the one
being trained on.


In [ ]:
dataset = relbench.load_dataset("rel-f1")
db = dataset.get_db()
col_to_stype = get_stype_proposal(db)
data, col_stats = make_pkey_fkey_graph(db, col_to_stype)

print(data)
print()
print("node types:", sorted(data.node_types))
print("edge types:", len(data.edge_types))
for et in sorted(data.edge_types)[:6]:
    print("   ", et, data[et].edge_index.shape)


In [ ]:
# Compare with the numbers measured locally (data/graph_stats.json).
import json
stats = json.load(open("data/graph_stats.json"))
mac_edges = {k for k in stats["edge_types"]}
gpu_edges = {f"{s} -{r}-> {d}" for s, r, d in data.edge_types}
print("edge types match:", mac_edges == gpu_edges)
print("local only:", sorted(mac_edges - gpu_edges)[:5])
print("gpu only:", sorted(gpu_edges - mac_edges)[:5])


## 5. Training

The script is called from the command line, with the same arguments as locally.


In [ ]:
# One run first, to see it train end to end.
!python src/train.py --task driver-top3 --seed 0 --epochs 20


## 6. All tasks, 3 seeds each

Results are appended to `output/evaluation.csv`, alongside the tabular rows.


In [ ]:
for task in ["driver-top3", "driver-dnf", "driver-position"]:
    for seed in [0, 1, 2]:
        !python src/train.py --task {task} --seed {seed} --epochs 20


## 7. Download the results


In [ ]:
import pandas as pd
d = pd.read_csv("output/evaluation.csv")
print(d.pivot_table(index=["task", "metric", "split"], columns="arm", values="value").round(4))
d.to_csv("/kaggle/working/evaluation.csv", index=False)
